# 06 — Conditional Distribution Analysis

How do precipitation distribution parameters shift under **seasonal**, **ENSO**, and **MJO** conditioning?  
This is Phase 1 of the subseasonal forecast project — the scientific foundation for the ML model target.

**Dataset:** CPC gauge-based only (2001–2020, 7-day means, wet threshold 0.1 mm/day)

**Three independent stratifications:**
| Condition | Strata | Approx. samples/stratum |
|---|---|---|
| Season | DJF / MAM / JJA / SON | ~260 ✅ |
| ENSO | El Niño / Neutral / La Niña | ~87 ⚠️ marginal |
| MJO | phases 1–8 (RMM ≥ 1.0) + inactive | ~30–60 ⚠️ assess |

**Index definitions:**
- **ENSO:** Niño 3.4 SST anomaly ±0.5°C, 3-month running mean (ONI; Johnson et al. 2014)
- **MJO:** RMM (Wheeler & Hendon 2004), amplitude ≥ 1.0, all 8 phases individually

**Kernel:** Python (atmo)

In [ ]:
import os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import yaml
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from src.analysis.conditional import (
    fit_conditional_distributions, load_conditional_stats,
    label_windows, SEASON_ORDER, ENSO_ORDER, MJO_ORDER,
)
from src.data.download_oni import download_oni, load_oni
from src.data.download_rmm import download_rmm, load_rmm

# ── Load config ───────────────────────────────────────────────────────────
with open('../config/config.yaml') as f:
    config = yaml.safe_load(f)

MIN_SAMPLES = config['analysis']['min_samples']

# ── Distribution colour convention (matches notebook 05) ─────────────────
DIST_NAMES  = ['Log-normal', 'Gamma', 'Pearson III', 'SHASH']
DIST_COLORS = ['#2166ac', '#d6604d', '#4dac26', '#f1a340']

# ── Cartopy helper ────────────────────────────────────────────────────────
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader

def _conus_ax(ax):
    ax.set_extent([-125, -66, 24, 50], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=4)
    countries = shpreader.natural_earth(resolution='50m', category='cultural',
                                        name='admin_0_countries')
    for rec in shpreader.Reader(countries).records():
        if rec.attributes.get('NAME_EN', '') in ('Canada', 'Mexico'):
            ax.add_geometries([rec.geometry], ccrs.PlateCarree(),
                              facecolor='lightgrey', edgecolor='none', zorder=4)
    ax.add_feature(cfeature.STATES, linewidth=0.4, edgecolor='black', alpha=0.6, zorder=5)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.6, zorder=5)
    return ax

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
print('Ready.')

---
## 1. Load CPC data and download index files

In [ ]:
# ── Processed CPC precipitation ────────────────────────────────────────────
da_cpc = xr.open_dataarray('../data/processed/cpc/cpc_7day_wet.nc')
print('CPC 7-day windows:', dict(da_cpc.sizes))
print('Time range:', str(da_cpc.time.values[0])[:10], '→', str(da_cpc.time.values[-1])[:10])

# ── Download index files (no-op if already cached) ────────────────────────
download_oni(config)
download_rmm(config)

# ── Load index DataFrames ─────────────────────────────────────────────────
oni_df = load_oni(config)
rmm_df = load_rmm(config)

print(f'\nONI: {len(oni_df)} monthly values ({oni_df.index[0].date()} → {oni_df.index[-1].date()})')
print('ENSO phase counts:')
print(oni_df['enso_phase'].value_counts().to_string())

print(f'\nRMM: {len(rmm_df)} daily values ({rmm_df.index[0].date()} → {rmm_df.index[-1].date()})')
active = rmm_df['mjo_active'].sum()
print(f'Active MJO days: {active} / {len(rmm_df)}  ({100*active/len(rmm_df):.1f}%)')

---
## 2. Sample counts per stratum

Before fitting, verify that each stratum has enough samples (≥ 30 wet-day windows per grid cell).  
This section shows **global counts** (number of 7-day windows assigned to each stratum).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, condition, strata, title in zip(
    axes,
    ['season', 'enso', 'mjo'],
    [SEASON_ORDER, ENSO_ORDER, MJO_ORDER],
    ['Season', 'ENSO phase', 'MJO phase'],
):
    kwargs = dict(condition=condition)
    if condition == 'enso': kwargs['oni_df'] = oni_df
    if condition == 'mjo':  kwargs['rmm_df'] = rmm_df; kwargs['config'] = config

    labels = label_windows(da_cpc, **kwargs)

    counts = [int((labels == s).sum()) for s in strata]
    colors = ['#4dac26' if c >= 30 else '#d62728' for c in counts]
    ax.bar(range(len(strata)), counts, color=colors, edgecolor='white')
    ax.axhline(30, color='red', ls='--', lw=1, label='min=30')
    ax.set_xticks(range(len(strata)))
    ax.set_xticklabels([s.replace('_', '\n') for s in strata], fontsize=8)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Number of 7-day windows')
    ax.legend(fontsize=8)
    for i, c in enumerate(counts):
        ax.text(i, c + 3, str(c), ha='center', fontsize=7)

fig.suptitle('Sample counts per conditioning stratum\n(green = adequate; red = below minimum)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Seasonal analysis

Re-fit 4 distributions at every CPC land cell using only windows from each season.  
Expected: P3 should still dominate, but the P3 skewness pattern should differ markedly  
between JJA (summer convective) and DJF (frontal/orographic).

In [ ]:
# Fit or load from cache (set overwrite=True to re-run)
season_results = fit_conditional_distributions(
    config, dataset='cpc', condition='season', workers=16, overwrite=False,
)
print('\nLoaded seasons:', list(season_results.keys()))

In [ ]:
# ── Best-fit distribution map by season (1×4 panel) ──────────────────────
fig, axes = plt.subplots(
    1, 4, figsize=(22, 5),
    subplot_kw={'projection': ccrs.PlateCarree()},
)
cmap4 = mcolors.ListedColormap(DIST_COLORS)
norm4 = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap4.N)
cmap4.set_bad('lightgrey')

for ax, season in zip(axes, SEASON_ORDER):
    ax = _conus_ax(ax)
    ds = season_results.get(season)
    if ds is None:
        ax.set_title(f'{season}\n(no data)', fontsize=10)
        continue
    data = ds['best_fit_4way'].values.astype(float)
    data[ds['n_wetdays'].values < MIN_SAMPLES] = np.nan
    im = ax.pcolormesh(ds.lon.values, ds.lat.values, data,
                       transform=ccrs.PlateCarree(), cmap=cmap4, norm=norm4)
    n_tot = int(np.isfinite(data).sum())
    title_parts = []
    for i, name in enumerate(DIST_NAMES):
        pct = 100 * int(np.nansum(data == i)) / max(n_tot, 1)
        if pct > 1:
            title_parts.append(f'{name.split()[0]} {pct:.0f}%')
    ax.set_title(f'{season}\n' + '  ·  '.join(title_parts), fontsize=9, fontweight='bold')

sm = plt.cm.ScalarMappable(cmap=cmap4, norm=norm4)
cb = fig.colorbar(sm, ax=axes, orientation='horizontal', pad=0.04, shrink=0.4, ticks=[0,1,2,3])
cb.ax.set_xticklabels(DIST_NAMES, fontsize=9)
fig.suptitle('Best-fit distribution by season (CPC, 2001–2020)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── P3 skewness by season (1×4 panel) ─────────────────────────────────────
fig, axes = plt.subplots(
    1, 4, figsize=(22, 5),
    subplot_kw={'projection': ccrs.PlateCarree()},
)
cmap_skew = plt.get_cmap('YlOrRd')
cmap_skew.set_bad('lightgrey')

for ax, season in zip(axes, SEASON_ORDER):
    ax = _conus_ax(ax)
    ds = season_results.get(season)
    if ds is None:
        ax.set_title(f'{season}\n(no data)', fontsize=10)
        continue
    data = ds['pearson3_skew'].values.copy()
    data[ds['n_wetdays'].values < MIN_SAMPLES] = np.nan
    data = np.clip(data, 0, 4)
    im = ax.pcolormesh(ds.lon.values, ds.lat.values, data,
                       transform=ccrs.PlateCarree(), cmap=cmap_skew, vmin=0, vmax=4)
    ax.set_title(season, fontsize=11, fontweight='bold')

cb = fig.colorbar(im, ax=axes, orientation='horizontal', pad=0.04, shrink=0.5)
cb.set_label('Pearson III skewness (0 = symmetric → 4 = strongly convective)', fontsize=9)
fig.suptitle('Pearson III skewness by season  (CPC, 2001–2020)\n'
             'Yellow = stratiform/orographic  ·  Red = convective/extreme-dominated',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. ENSO analysis

Re-fit using windows stratified by ENSO phase (El Niño / Neutral / La Niña).  
**Expected signals** (from Arcodia et al. 2020, Johnson et al. 2014):
- El Niño → wetter SE/SW, drier Pacific NW in DJF
- La Niña → wetter Pacific NW, drier SE/Texas in cool season

Key question: does the distribution *shape* (skewness, tail-weight) change, or just the scale?

In [ ]:
enso_results = fit_conditional_distributions(
    config, dataset='cpc', condition='enso', workers=16, overwrite=False,
)
print('Loaded ENSO strata:', list(enso_results.keys()))

In [ ]:
# ── P3 skewness by ENSO phase (1×3 panel) ────────────────────────────────
enso_labels = {'el_nino': 'El Niño', 'neutral': 'Neutral', 'la_nina': 'La Niña'}

fig, axes = plt.subplots(
    1, 3, figsize=(18, 5),
    subplot_kw={'projection': ccrs.PlateCarree()},
)
cmap_skew = plt.get_cmap('YlOrRd')
cmap_skew.set_bad('lightgrey')

for ax, phase in zip(axes, ENSO_ORDER):
    ax = _conus_ax(ax)
    ds = enso_results.get(phase)
    if ds is None:
        ax.set_title(f'{enso_labels[phase]}\n(no data)', fontsize=10)
        continue
    data = ds['pearson3_skew'].values.copy()
    data[ds['n_wetdays'].values < MIN_SAMPLES] = np.nan
    data = np.clip(data, 0, 4)
    im = ax.pcolormesh(ds.lon.values, ds.lat.values, data,
                       transform=ccrs.PlateCarree(), cmap=cmap_skew, vmin=0, vmax=4)
    n_win = ds.attrs.get('n_windows', '?')
    ax.set_title(f"{enso_labels[phase]}  (n={n_win})", fontsize=11, fontweight='bold')

cb = fig.colorbar(im, ax=axes, orientation='horizontal', pad=0.04, shrink=0.4)
cb.set_label('Pearson III skewness [0–4]', fontsize=9)
fig.suptitle('Pearson III skewness by ENSO phase  (CPC, 2001–2020)\n'
             'Niño 3.4 ±0.5°C, 3-month running mean (ONI)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. MJO phase analysis

Re-fit using windows stratified by MJO phase.  
**Expected signals** (Schreck et al. 2021, Zhou et al. 2012, Mundhenk et al. 2018):
- Phases 2–3 → SE US wet, Pacific NW dry
- Phases 6–7 → SE US dry, West Coast (Oct–Dec) wet
- Phases 4–5 → Central Plains wet

**Note on sample sizes:** Each MJO phase stratum has ~30–60 windows.  
Cells with fewer than 30 wet-day samples within a stratum will be masked.

In [ ]:
mjo_results = fit_conditional_distributions(
    config, dataset='cpc', condition='mjo', workers=16, overwrite=False,
)
print('Loaded MJO strata:', list(mjo_results.keys()))

In [ ]:
# ── P3 skewness by MJO phase (2×4 panel — phases 1–8) ────────────────────
active_phases = [f'phase_{i}' for i in range(1, 9)]

fig, axes = plt.subplots(
    2, 4, figsize=(22, 9),
    subplot_kw={'projection': ccrs.PlateCarree()},
)
cmap_skew = plt.get_cmap('YlOrRd')
cmap_skew.set_bad('lightgrey')

for ax, phase_key in zip(axes.flat, active_phases):
    ax = _conus_ax(ax)
    ds = mjo_results.get(phase_key)
    phase_num = phase_key.split('_')[1]
    if ds is None:
        ax.set_title(f'MJO Phase {phase_num}\n(no data / insufficient n)', fontsize=9)
        continue
    data = ds['pearson3_skew'].values.copy()
    data[ds['n_wetdays'].values < MIN_SAMPLES] = np.nan
    data = np.clip(data, 0, 4)
    im = ax.pcolormesh(ds.lon.values, ds.lat.values, data,
                       transform=ccrs.PlateCarree(), cmap=cmap_skew, vmin=0, vmax=4)
    n_win = ds.attrs.get('n_windows', '?')
    ax.set_title(f'MJO Phase {phase_num}  (n={n_win})', fontsize=9, fontweight='bold')

cb = fig.colorbar(im, ax=axes, orientation='horizontal', pad=0.04, shrink=0.4)
cb.set_label('Pearson III skewness [0–4]', fontsize=9)
fig.suptitle(
    'Pearson III skewness by MJO phase  (CPC, 2001–2020, active MJO: RMM ≥ 1.0)\n'
    'Phases 2–3: SE wet  ·  Phases 6–7: SE dry / W-Coast wet  ·  Phases 4–5: Central Plains wet',
    fontsize=12, fontweight='bold', y=1.01,
)
plt.tight_layout()
plt.show()